In [6]:
import ccxt
import pandas as pd
import numpy as np
import ta

from tabulate import tabulate

In [7]:
exchange = ccxt.binance({
    "enableRateLimit": True,
    "options": {
        "defaultType": "future"   # ← THIS switches to futures
    }
})

In [8]:
markets = exchange.load_markets()

usdt_pairs = [
    symbol for symbol, market in markets.items()
    if market['quote'] == 'USDT'     # Must be USDT pair
    and market['contract']           # Must be derivative
    and market['swap']               # Must be perpetual (not quarterly)
    and market['active']             # Must be active
]

print(f"Total active USDT futures pairs: {len(usdt_pairs)}")
print(usdt_pairs[:10])


Total active USDT futures pairs: 579
['BTC/USDT:USDT', 'ETH/USDT:USDT', 'BCH/USDT:USDT', 'XRP/USDT:USDT', 'LTC/USDT:USDT', 'TRX/USDT:USDT', 'ETC/USDT:USDT', 'LINK/USDT:USDT', 'XLM/USDT:USDT', 'ADA/USDT:USDT']


In [9]:
filtered_pairs = []

for symbol in usdt_pairs:
    print(f"scan {symbol}")
    ticker = exchange.fetch_ticker(symbol)

    if ticker['quoteVolume'] and ticker['quoteVolume'] > 5_000_000:
        filtered_pairs.append(symbol)

print(f"High-liquidity USDT futures: {len(filtered_pairs)}")


scan BTC/USDT:USDT
scan ETH/USDT:USDT
scan BCH/USDT:USDT
scan XRP/USDT:USDT
scan LTC/USDT:USDT
scan TRX/USDT:USDT
scan ETC/USDT:USDT
scan LINK/USDT:USDT
scan XLM/USDT:USDT
scan ADA/USDT:USDT
scan XMR/USDT:USDT
scan DASH/USDT:USDT
scan ZEC/USDT:USDT
scan XTZ/USDT:USDT
scan BNB/USDT:USDT
scan ATOM/USDT:USDT
scan ONT/USDT:USDT
scan IOTA/USDT:USDT
scan BAT/USDT:USDT
scan VET/USDT:USDT
scan NEO/USDT:USDT
scan QTUM/USDT:USDT
scan IOST/USDT:USDT
scan THETA/USDT:USDT
scan ALGO/USDT:USDT
scan ZIL/USDT:USDT
scan KNC/USDT:USDT
scan ZRX/USDT:USDT
scan COMP/USDT:USDT
scan DOGE/USDT:USDT
scan KAVA/USDT:USDT
scan BAND/USDT:USDT
scan RLC/USDT:USDT
scan SNX/USDT:USDT
scan DOT/USDT:USDT
scan YFI/USDT:USDT
scan CRV/USDT:USDT
scan TRB/USDT:USDT
scan RUNE/USDT:USDT
scan SUSHI/USDT:USDT
scan EGLD/USDT:USDT
scan SOL/USDT:USDT
scan ICX/USDT:USDT
scan STORJ/USDT:USDT
scan UNI/USDT:USDT
scan AVAX/USDT:USDT
scan ENJ/USDT:USDT
scan KSM/USDT:USDT
scan NEAR/USDT:USDT
scan AAVE/USDT:USDT
scan FIL/USDT:USDT
scan RSR/

In [10]:
def get_ohlcv(symbol, timeframe='15m', limit=200):
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)

    df = pd.DataFrame(ohlcv, columns=[
        'timestamp','open','high','low','close','volume'
    ])

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df


In [11]:
import pandas as pd
import numpy as np

def add_rsi_tradingview(df, length=14, source_col='close'):
    """
    TradingView / Pine Script RSI implementation using RMA (Wilder)
    """

    # 1. Price change (ta.change)
    change = df[source_col].diff()

    # 2. Up & Down series
    up = change.clip(lower=0)
    down = -change.clip(upper=0)

    # 3. Wilder's Moving Average (RMA)
    alpha = 1 / length
    avg_up = up.ewm(alpha=alpha, adjust=False).mean()
    avg_down = down.ewm(alpha=alpha, adjust=False).mean()

    # 4. RSI calculation
    rs = avg_up / avg_down
    rsi = np.where(
        avg_down == 0, 100,
        np.where(avg_up == 0, 0, 100 - (100 / (1 + rs)))
    )

    df['rsi'] = rsi
    return df


In [12]:
def add_stochastic_tv(df, periodK=14, smoothK=3, periodD=3):
    
    # Raw stochastic
    low_min = df['low'].rolling(periodK).min()
    high_max = df['high'].rolling(periodK).max()
    
    stoch = 100 * (df['close'] - low_min) / (high_max - low_min)
    
    # Smooth K
    k = stoch.rolling(smoothK).mean()
    
    # Smooth D
    d = k.rolling(periodD).mean()
    
    df['stoch_k'] = k
    df['stoch_d'] = d
    
    return df

In [13]:
def add_stoch_rsi_tv(df, lengthRSI=14, lengthStoch=14, smoothK=3, smoothD=3):

    # --- TRUE TradingView RSI (RMA based)
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    alpha = 1 / lengthRSI
    avg_gain = gain.ewm(alpha=alpha, adjust=False).mean()
    avg_loss = loss.ewm(alpha=alpha, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    # --- TRUE Stochastic of RSI
    rsi_min = rsi.rolling(lengthStoch).min()
    rsi_max = rsi.rolling(lengthStoch).max()

    stoch = 100 * (rsi - rsi_min) / (rsi_max - rsi_min)

    # --- Smooth K and D
    k = stoch.rolling(smoothK).mean()
    d = k.rolling(smoothD).mean()

    df['stoch_rsi_k'] = k
    df['stoch_rsi_d'] = d

    return df

In [14]:
def add_ma99(df):

    df['ma99'] = ta.trend.ema_indicator(
        close=df['close'],
        window=99
    )

    return df

In [15]:
def check_signal(df):
    last = df.iloc[-1]

    rsi = last["rsi"]
    stoch_rsi = last["stoch_rsi_k"]
    stoch_k = last["stoch_d"]

    # =========================
    # LONG CONDITIONS
    # =========================
    oversold = (
        rsi < 45 and
        stoch_rsi < 45 and
        stoch_k < 45
    )

    momentum_up = stoch_k > stoch_rsi

    long_signal = oversold and momentum_up

    # =========================
    # SHORT CONDITIONS (OPPOSITE)
    # =========================
    overbought = (
        rsi > 55 and
        stoch_rsi > 55 and
        stoch_k > 55
    )

    momentum_down = stoch_k < stoch_rsi

    short_signal = overbought and momentum_down

    if long_signal:
        return "LONG"
    elif short_signal:
        return "SHORT"
    else:
        return None


In [ ]:
def check_4h_trend(df):

    last = df.iloc[-1]

    close = last['close']

    ma99 = last['ma99']

    rsi = last['rsi']

    stoch_k = last['stoch_d']

    stoch_rsi = last['stoch_rsi_k']

    # =====================================
    # BULLISH TREND
    # =====================================

    bullish = (

        close > ma99

        and rsi > 50

        and stoch_k > 50

        and stoch_rsi > 50

        and stoch_k > stoch_rsi
    )

    # =====================================
    # BEARISH TREND
    # =====================================

    bearish = (

        close < ma99

        and rsi < 50

        and stoch_k < 50

        and stoch_rsi < 50

        and stoch_k < stoch_rsi
    )

    if bullish:

        return "BULLISH"

    elif bearish:

        return "BEARISH"

    return None

In [17]:
def analyze_symbol(symbol):

    try:

        df = get_ohlcv(symbol, '4h', 500)

        # remove open candle
        df = df.iloc[:-1]

        df = add_rsi_tradingview(df)

        df = add_stochastic_tv(df)

        df = add_stoch_rsi_tv(df)

        df = add_ma99(df)

        signal = check_4h_trend(df)

        if signal:

            return {

                'Symbol': symbol,

                'Trend': signal,

                'Price': round(df.iloc[-1]['close'], 4),

                'RSI': round(df.iloc[-1]['rsi'], 2)
            }

    except Exception as e:

        print(f"Error {symbol}: {e}")

    return None

In [18]:
def scan_market(pairs):

    results = []

    for i, symbol in enumerate(pairs, 1):

        print(f"[{i}/{len(pairs)}] Scanning {symbol}")

        result = analyze_symbol(symbol)

        if result:

            print(f"FOUND {result['Trend']} → {symbol}")

            results.append(result)

    return results

In [19]:
signals = scan_market(filtered_pairs)

[1/273] Scanning BTC/USDT:USDT
FOUND BEARISH → BTC/USDT:USDT
[2/273] Scanning ETH/USDT:USDT
FOUND BEARISH → ETH/USDT:USDT
[3/273] Scanning BCH/USDT:USDT
[4/273] Scanning XRP/USDT:USDT
FOUND BEARISH → XRP/USDT:USDT
[5/273] Scanning LTC/USDT:USDT
FOUND BEARISH → LTC/USDT:USDT
[6/273] Scanning TRX/USDT:USDT
[7/273] Scanning ETC/USDT:USDT
FOUND BEARISH → ETC/USDT:USDT
[8/273] Scanning LINK/USDT:USDT
[9/273] Scanning XLM/USDT:USDT
[10/273] Scanning ADA/USDT:USDT
FOUND BEARISH → ADA/USDT:USDT
[11/273] Scanning XMR/USDT:USDT
FOUND BEARISH → XMR/USDT:USDT
[12/273] Scanning DASH/USDT:USDT
FOUND BEARISH → DASH/USDT:USDT
[13/273] Scanning ZEC/USDT:USDT
[14/273] Scanning BNB/USDT:USDT
[15/273] Scanning ATOM/USDT:USDT
[16/273] Scanning ONT/USDT:USDT
FOUND BEARISH → ONT/USDT:USDT
[17/273] Scanning VET/USDT:USDT
[18/273] Scanning THETA/USDT:USDT
[19/273] Scanning ALGO/USDT:USDT
FOUND BEARISH → ALGO/USDT:USDT
[20/273] Scanning COMP/USDT:USDT
FOUND BEARISH → COMP/USDT:USDT
[21/273] Scanning DOGE/USDT:U

In [20]:
results_df = pd.DataFrame(signals)

results_df = results_df.sort_values(
    by='RSI',
    ascending=False
)

print(
    tabulate(
        results_df,
        headers='keys',
        tablefmt='fancy_grid',
        showindex=False
    )
)

╒═════════════════════╤═════════╤════════════╤═══════╕
│ Symbol              │ Trend   │      Price │   RSI │
╞═════════════════════╪═════════╪════════════╪═══════╡
│ IN/USDT:USDT        │ BULLISH │     0.0787 │ 92.27 │
├─────────────────────┼─────────┼────────────┼───────┤
│ BEAT/USDT:USDT      │ BULLISH │     1.2365 │ 91.07 │
├─────────────────────┼─────────┼────────────┼───────┤
│ USDC/USDT:USDT      │ BULLISH │     1.0003 │ 83.37 │
├─────────────────────┼─────────┼────────────┼───────┤
│ QCOM/USDT:USDT      │ BULLISH │   237.68   │ 77.71 │
├─────────────────────┼─────────┼────────────┼───────┤
│ GENIUS/USDT:USDT    │ BULLISH │     0.6261 │ 75.53 │
├─────────────────────┼─────────┼────────────┼───────┤
│ JCT/USDT:USDT       │ BULLISH │     0.004  │ 75.13 │
├─────────────────────┼─────────┼────────────┼───────┤
│ TAG/USDT:USDT       │ BULLISH │     0.0015 │ 73.37 │
├─────────────────────┼─────────┼────────────┼───────┤
│ NIGHT/USDT:USDT     │ BULLISH │     0.0332 │ 65.72 │
├─────────